# Glossary check: F4 Darija–French with a connector glossary (non-thinking)

Reruns the F4 Darija–French cases (base and contrast, formats A and B) with your glossary of Darija time connectors added just before the story. Same model, revision, decoding and answer parsing as the main run; only the glossary is new.

1. Runtime → **T4 GPU**. 2. Upload `cases.csv` and `glossary.txt` (both from `data/`). 3. Run all. 4. Download `glossary_results.zip`.

In [ ]:
!pip install -q -U "transformers>=5.0" accelerate torchvision pillow

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-4B"
REVISION = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"  # main on 2026-10-21; pinned for reproducibility
CASES_PATH = "cases.csv"
OUT_DIR = "results"
FORMATS = ["A", "B"]
MAX_NEW_TOKENS = 384          # non-thinking answers are short; truncation is flagged per item
THINKING_MAX_NEW_TOKENS = 4096

In [ ]:
OUT_DIR = "results_glossary"  # separate folder: never overwrite the main run results

In [ ]:
# --- scripts/pilot.py, embedded by scripts/build_notebook.py ---
"""Case loading, prompt building, answer parsing and metrics for the pilot.

The notebook embeds this file's source in one cell (scripts/build_notebook.py), so
Colab needs no repo clone. Model loading and generation live in the notebook.
"""
import csv
import json
import re
from collections import defaultdict

STATUS_LABELS = ("CONFIRMED_ACTIVE", "CONFIRMED_NOT_ACTIVE", "INSUFFICIENT_EVIDENCE")
TASK_TYPES = ("status", "order", "extract")
LANGUAGES = ("fr", "darija_fr")
CASE_COLUMNS = (
    "family_id", "variant_id", "language", "task_type", "fictional_rule", "narrative",
    "question", "gold_answer", "gold_rationale", "changed_feature", "same_answer_as_base",
    "written_by", "review_status", "notes",
)

ANSWER_INSTRUCTIONS = {
    "status": "End your reply with one line: ANSWER: CONFIRMED_ACTIVE, ANSWER: CONFIRMED_NOT_ACTIVE or ANSWER: INSUFFICIENT_EVIDENCE.",
    "order": "End your reply with one line: ANSWER: followed by the event IDs in chronological order, comma-separated (for example ANSWER: E2, E1, E3).",
    "extract": "End your reply with one line: ANSWER: followed by your short answer.",
}

# Two fixed layouts with identical content; only formatting differs (prompt-format check).
PROMPT_FORMATS = {
    "A": "Rule: {rule}\n\nStory: {narrative}\n\nQuestion: {question}\n\n{instruction}",
    "B": "### RULE\n{rule}\n### STORY\n{narrative}\n### QUESTION\n{question}\n### FORMAT\n{instruction}",
}


def case_id(case):
    return f"{case['family_id']}-{case['variant_id']}-{case['language']}"


def load_cases(path, approved_only=True):
    """Read the review sheet CSV; keep APPROVED/EDITED rows unless approved_only is False."""
    with open(path, encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    missing = [c for c in CASE_COLUMNS if rows and c not in rows[0]]
    if missing:
        raise ValueError(f"cases file is missing columns: {missing}")
    cases, problems = [], []
    for n, row in enumerate(rows, start=2):
        row = {k: (v or "").strip() for k, v in row.items() if k}
        if not row.get("family_id"):
            continue
        if approved_only and row["review_status"].upper() not in ("APPROVED", "EDITED"):
            continue
        if row["task_type"] not in TASK_TYPES:
            problems.append(f"line {n}: task_type {row['task_type']!r} not in {TASK_TYPES}")
        if row["language"] not in LANGUAGES:
            problems.append(f"line {n}: language {row['language']!r} not in {LANGUAGES}")
        if row["task_type"] == "status" and row["gold_answer"] not in STATUS_LABELS:
            problems.append(f"line {n}: status gold {row['gold_answer']!r} not in {STATUS_LABELS}")
        row["case_id"] = case_id(row)
        cases.append(row)
    ids = [c["case_id"] for c in cases]
    dupes = sorted({i for i in ids if ids.count(i) > 1})
    if dupes:
        problems.append(f"duplicate case ids: {dupes}")
    if problems:
        raise ValueError("cases file problems:\n" + "\n".join(problems))
    return cases


GLOSSARY_BLOCKS = {
    "A": "Glossary of Darija words: {glossary}\n\nStory:",
    "B": "### GLOSSARY OF DARIJA WORDS\n{glossary}\n### STORY",
}


def build_prompt(case, fmt, glossary=None):
    """With a glossary, the same prompt gets a glossary section just before the story."""
    prompt = PROMPT_FORMATS[fmt].format(
        rule=case["fictional_rule"], narrative=case["narrative"], question=case["question"],
        instruction=ANSWER_INSTRUCTIONS[case["task_type"]],
    )
    if glossary:
        marker = "Story:" if fmt == "A" else "### STORY"
        prompt = prompt.replace(marker, GLOSSARY_BLOCKS[fmt].format(glossary=glossary.strip()), 1)
    return prompt


def load_glossary(path):
    text = open(path, encoding="utf-8-sig").read().strip()
    lines = [l for l in text.splitlines() if l.strip() and not l.lstrip().startswith("#")]
    if not lines or any("TODO" in l for l in lines):
        raise ValueError(f"{path} still has TODO lines or is empty; write the glossary first")
    return "\n".join(lines)


def final_text(raw):
    """Drop any thinking block; only the text after </think> is scored."""
    return raw.rsplit("</think>", 1)[-1].strip()


def parse_answer(raw, task_type):
    """Return (parsed_answer, parse_status). parse_status: ok | no_answer_line | invalid."""
    text = final_text(raw)
    lines = [l for l in text.splitlines() if re.match(r"\s*\**\s*ANSWER\s*\**\s*:", l, re.I)]
    if not lines:
        return "", "no_answer_line"
    value = lines[-1].split(":", 1)[1].strip().strip("*").strip()
    if task_type == "status":
        found = [lab for lab in STATUS_LABELS if re.search(rf"\b{lab}\b", value.upper())]
        return (found[0], "ok") if len(found) == 1 else (value, "invalid")
    if task_type == "order":
        ids = re.findall(r"\bE\d+\b", value.upper())
        return (", ".join(ids), "ok") if ids else (value, "invalid")
    return value, "ok"


def normalise_order(s):
    return [x for x in re.findall(r"E\d+", s.upper())]


def auto_score(case, parsed, parse_status):
    """True/False when checkable automatically, None when a human must judge (extract tasks)."""
    if parse_status != "ok":
        return False
    if case["task_type"] == "status":
        return parsed == case["gold_answer"]
    if case["task_type"] == "order":
        return normalise_order(parsed) == normalise_order(case["gold_answer"])
    return None


def final_correct(row):
    """The human verdict wins over the automatic score."""
    manual = (row.get("manual_verdict") or "").strip().upper()
    if manual in ("CORRECT", "WRONG"):
        return manual == "CORRECT"
    auto = row.get("auto_correct")
    if auto in (True, False):
        return auto
    if isinstance(auto, str) and auto in ("True", "False"):
        return auto == "True"
    return None


def _ratio(rows):
    judged = [r for r in rows if final_correct(r) is not None]
    return f"{sum(final_correct(r) for r in judged)}/{len(judged)}"


def metrics(rows):
    """Counts as numerator/denominator strings; unjudged rows are left out of each ratio."""
    out = {"overall": _ratio(rows), "unjudged": sum(final_correct(r) is None for r in rows)}
    for key in ("language", "prompt_format", "task_type", "variant_id"):
        groups = defaultdict(list)
        for r in rows:
            groups[r[key]].append(r)
        out[f"by_{key}"] = {k: _ratio(v) for k, v in sorted(groups.items())}
    out["parse_failures"] = sum(r["parse_status"] != "ok" for r in rows)

    by_id = {(r["family_id"], r["variant_id"], r["language"], r["prompt_format"]): r for r in rows}
    # Cross-language consistency: same case and format, French vs Darija-French.
    same, both_right, pairs = 0, 0, 0
    for (fam, var, lang, fmt), r in by_id.items():
        other = by_id.get((fam, var, "darija_fr", fmt)) if lang == "fr" else None
        if not other:
            continue
        a, b = final_correct(r), final_correct(other)
        if a is None or b is None:
            continue
        pairs += 1
        same += (r["parsed_answer"].upper() == other["parsed_answer"].upper())
        both_right += a and b
    out["cross_language_same_answer"] = f"{same}/{pairs}"
    out["cross_language_both_correct"] = f"{both_right}/{pairs}"

    # Contrast consistency: base and contrast both correct, per family, language and format.
    groups, ok = 0, 0
    for (fam, var, lang, fmt), r in by_id.items():
        if var != "base":
            continue
        twin = by_id.get((fam, "contrast", lang, fmt))
        if not twin or final_correct(r) is None or final_correct(twin) is None:
            continue
        groups += 1
        ok += final_correct(r) and final_correct(twin)
    out["contrast_consistency"] = f"{ok}/{groups}"
    return out


def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path):
    try:
        with open(path, encoding="utf-8") as f:
            return [json.loads(l) for l in f if l.strip()]
    except FileNotFoundError:
        return []

In [ ]:
import os, sys, json, time, platform, datetime
import torch, transformers
os.makedirs(OUT_DIR, exist_ok=True)
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
assert gpu, "No GPU: Runtime -> Change runtime type -> T4 GPU"
# T4 (compute 7.5) has no native bf16; use float16 there.
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
run_config = {
    "model_id": MODEL_ID, "revision": REVISION, "gpu": gpu, "dtype": str(DTYPE),
    "python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
    "decoding": {"do_sample": False, "num_beams": 1, "max_new_tokens": MAX_NEW_TOKENS},
    "prompt_formats": PROMPT_FORMATS, "answer_instructions": ANSWER_INSTRUCTIONS,
    "started_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),
}
json.dump(run_config, open(f"{OUT_DIR}/run_config.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps(run_config, indent=2, ensure_ascii=False))

In [ ]:
GLOSSARY = load_glossary("glossary.txt")
print(GLOSSARY)
cases = [c for c in load_cases(CASES_PATH) if c["family_id"] == "F4" and c["language"] == "darija_fr"]
print(len(cases), "F4 Darija cases")

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, revision=REVISION, dtype=DTYPE, device_map="auto")
model.eval()

def render(prompt, thinking):
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    return processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False,
                                         enable_thinking=thinking)

def generate(prompt, thinking=False, max_new_tokens=MAX_NEW_TOKENS):
    text = render(prompt, thinking)
    inputs = processor(text=[text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, do_sample=False, num_beams=1, max_new_tokens=max_new_tokens)
    new = out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(new, skip_special_tokens=True), len(new)

# Check the chat template really switched thinking off: the prompt should end with an empty think block.
sample = render(build_prompt(cases[0], "A"), thinking=False)
print(sample[-300:])
assert "</think>" in sample[-60:], "enable_thinking=False did not reach the chat template; stop and check"

In [ ]:
RAW_G = f"{OUT_DIR}/raw_outputs_glossary.jsonl"
json.dump({**run_config, "glossary": GLOSSARY, "cases": [c["case_id"] for c in cases]},
          open(f"{OUT_DIR}/run_config_glossary.json", "w"), indent=2, ensure_ascii=False)
done = {(r["case_id"], r["prompt_format"]) for r in read_jsonl(RAW_G)}
todo = [(c, f) for f in FORMATS for c in cases if (c["case_id"], f) not in done]
print(f"{len(done)} done, {len(todo)} to run")
for k, (c, fmt) in enumerate(todo, 1):
    prompt = build_prompt(c, fmt, glossary=GLOSSARY)
    t = time.time()
    raw, n_tokens = generate(prompt)
    parsed, status = parse_answer(raw, c["task_type"])
    append_jsonl(RAW_G, {"case_id": c["case_id"], "prompt_format": fmt, "thinking": False, "glossary": True,
                         "prompt": prompt, "raw_output": raw, "new_tokens": n_tokens,
                         "truncated": n_tokens >= MAX_NEW_TOKENS, "parsed_answer": parsed, "parse_status": status,
                         "auto_correct": auto_score(c, parsed, status), "seconds": round(time.time() - t, 1)})
    print(f"[{k}/{len(todo)}] {c['case_id']} fmt={fmt} -> {parsed or status} gold={c['gold_answer']} ({time.time() - t:.0f}s)")

In [ ]:
res = read_jsonl(RAW_G)
print(f"with glossary: {sum(r['auto_correct'] is True for r in res)}/{len(res)} correct (without glossary: 0/4 in the main run)")
for r in res:
    print("=====", r["case_id"], r["prompt_format"], "->", r["parsed_answer"])
    print(r["raw_output"][:1200])

In [ ]:
import shutil
shutil.copy("glossary.txt", f"{OUT_DIR}/glossary_used.txt")
shutil.make_archive("glossary_results", "zip", OUT_DIR)
from google.colab import files
files.download("glossary_results.zip")